# Fitbit / Bellabeat Case Study: Exploratory Data Analysis (EDA)

**Notebook:** `notebooks/eda.ipynb`  |  **Stage 2 of 4** (Cleaning → **EDA** → SQL analysis → Streamlit + report)

## Goal
Explore the cleaned tables visually to find patterns in **daily activity, sleep and BMI**, using a mix of chart types so the same kind of insight isn't always shown the same way.

## Input
This notebook reads only from `Data/processed/` (the CSVs saved by `data_cleaning.ipynb`). It never touches the raw files. If any of the 4 CSVs are missing, re-run the cleaning notebook first.

## Chart plan (20 questions, no chart type repeated back-to-back)

| # | Question | Chart |
|---|---|---|
| 1 | Steps distribution | Histogram |
| 2 | Calories distribution | Histogram |
| 3 | Steps vs calories | Scatter + regression line |
| 4 | Avg steps by weekday | Bar |
| 5 | Intensity minutes by weekday | Stacked bar |
| 6 | Share of time in each intensity | Donut |
| 7 | Steps trend over the month | Line (5 sample users) |
| 8 | Spread of active minutes | Box plot |
| 9 | Sleep duration distribution | Histogram |
| 10 | Time asleep vs time in bed | Scatter |
| 11 | Sleep: weekday vs weekend | Violin plot |
| 12 | Sleep vs next-day steps | Scatter |
| 13 | BMI distribution (per user) | Histogram |
| 14 | BMI category breakdown | Pie |
| 15 | Weight trend per user | Line |
| 16 | Correlation: steps, calories, active minutes, sleep, BMI | Heatmap |
| 17 | Hourly step pattern | Area/line |
| 18 | Weekday vs weekend hourly pattern | Grouped line |
| 19 | Engagement gap (users per data type) | Bar |
| 20 | Most vs least active users | Horizontal bar |

## Step 1: Imports and load the cleaned data

**What this does:** loads plotting libraries and the 4 cleaned CSVs written by the cleaning notebook.

**How it works**
- `matplotlib` draws the charts, `seaborn` sits on top of it for nicer statistical plots (histograms, violin plots, heatmaps).
- `sns.set_style("whitegrid")` gives every chart a light grid background, so it's easier to read values.
- `parse_dates=[...]` tells pandas to read those columns as real dates immediately, instead of doing it in every cell.
- `daily_worn` keeps only rows where `is_worn` is `True` — i.e. days the tracker actually recorded activity. Use `daily_worn` for any steps/calories/intensity chart, and the full `daily` table only if you specifically want to include not-worn days.

**Expected output:** `daily_worn (868, 20)` — the 72 not-worn days are excluded.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

PROCESSED = "../Data/processed"

daily = pd.read_csv(f"{PROCESSED}/daily_activity.csv", parse_dates=["date"])
sleep = pd.read_csv(f"{PROCESSED}/sleep_daily.csv", parse_dates=["date"])
weight = pd.read_csv(f"{PROCESSED}/weight_log.csv", parse_dates=["date"])
hourly = pd.read_csv(f"{PROCESSED}/hourly_activity.csv", parse_dates=["date", "datetime"])

daily_worn = daily[daily["is_worn"]]
print("daily_worn:", daily_worn.shape)

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## Step 2: Steps and calories distributions (Q1-Q2 — Histograms)

**What this does:** shows the overall shape of daily steps and daily calories — are most days clustered around a typical value, or spread out?

**How it works:** `sns.histplot` bins the values and draws bars, and `kde=True` overlays a smooth curve showing the same shape.

**What to look for:** a peak roughly around 6,000-8,000 steps is expected for this dataset, with a long right tail (a few very active days).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(daily_worn["TotalSteps"], bins=20, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Distribution of Daily Steps")
axes[0].set_xlabel("Total Steps")

sns.histplot(daily_worn["Calories"], bins=20, kde=True, ax=axes[1], color="#DD8452")
axes[1].set_title("Distribution of Daily Calories Burned")
axes[1].set_xlabel("Calories")

plt.tight_layout()
plt.show()

## Step 3: Steps vs calories (Q3 — Scatter with regression line)

**What this does:** checks whether walking more actually burns more calories in this data.

**How it works:** `sns.regplot` draws each day as a point and fits a straight trend line through them. `scatter_kws={"alpha":0.4}` makes points semi-transparent so overlapping days are visible.

**Finding:** correlation is about **0.57** — a moderate positive relationship. It is not stronger because calories also depend on things this dataset does not capture well, like body weight and non-step activity (e.g. cycling).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(data=daily_worn, x="TotalSteps", y="Calories",
            scatter_kws={"alpha": 0.4}, line_kws={"color": "red"}, ax=ax)
ax.set_title("Steps vs Calories Burned")
plt.show()

print("Correlation (steps, calories):", round(daily_worn["TotalSteps"].corr(daily_worn["Calories"]), 3))

## Step 4: Average steps by weekday (Q4 — Bar chart)

**What this does:** checks whether people walk more on certain days.

**How it works:** `groupby("day_of_week")` averages steps per weekday, and `.reindex(order)` puts the bars in Monday-to-Sunday order instead of alphabetical.

**Finding:** Tuesday and Saturday are the highest (about 8,800-8,900 steps), Sunday is the lowest (about 7,600) — a small weekday/weekend dip on Sunday specifically, not weekends overall.

In [ ]:
avg_steps_dow = daily_worn.groupby("day_of_week")["TotalSteps"].mean().reindex(order)

fig, ax = plt.subplots(figsize=(8, 4))
avg_steps_dow.plot(kind="bar", ax=ax, color="#55A868")
ax.set_title("Average Steps by Day of Week")
ax.set_ylabel("Average Steps")
ax.set_xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 5: Intensity minutes by weekday (Q5 — Stacked bar chart)

**What this does:** breaks each weekday down into how many minutes, on average, were sedentary / lightly / fairly / very active.

**How it works:** grouping by weekday and averaging the 4 intensity columns gives one row per weekday; `kind="bar", stacked=True` stacks the 4 categories in each bar instead of placing them side by side.

**What to look for:** sedentary minutes dominate every day (expected — most of a 24-hour day is not "active" time), so it's the *relative* size of the other 3 segments that matters more than the total bar height.

In [ ]:
intensity_cols = ["SedentaryMinutes", "LightlyActiveMinutes", "FairlyActiveMinutes", "VeryActiveMinutes"]
avg_intensity = daily_worn.groupby("day_of_week")[intensity_cols].mean().reindex(order)

fig, ax = plt.subplots(figsize=(9, 5))
avg_intensity.plot(kind="bar", stacked=True, ax=ax,
                    color=["#B0B0B0", "#8FBBD9", "#F1A340", "#D7301F"])
ax.set_title("Average Minutes by Intensity, per Weekday")
ax.set_ylabel("Minutes")
ax.legend(title="Intensity")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 6: Share of total time by intensity (Q6 — Donut chart)

**What this does:** shows what fraction of *all* tracked time, across every user and day, falls into each intensity category.

**How it works:** summing each intensity column across the whole table gives 4 totals; `ax.pie(..., wedgeprops=dict(width=0.4))` draws a pie with the middle cut out, making it a donut.

**Finding:** roughly **80%** of tracked time is sedentary, about **17%** lightly active, and very/fairly active together are only a few percent — a useful, blunt way to show how little of the day is spent actively moving.

In [ ]:
intensity_totals = daily_worn[intensity_cols].sum()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(intensity_totals, labels=intensity_totals.index, autopct="%1.1f%%",
       wedgeprops=dict(width=0.4), colors=["#B0B0B0", "#8FBBD9", "#F1A340", "#D7301F"])
ax.set_title("Share of Total Time by Intensity Category")
plt.show()

## Step 7: Steps trend over the month (Q7 — Line chart)

**What this does:** shows day-by-day step counts across the month, so you can see consistency vs. spikes/drop-offs.

**How it works:** plotting all 33 users on one chart would be unreadable, so this samples 5 users and draws one line per user, with `marker='o'` to show individual data points.

**Note:** in your own version, feel free to swap the 5 user IDs printed by `daily_worn["Id"].unique()[:5]` for specific ones you want to highlight (e.g. the most and least active from Step 20).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sample_users = daily_worn["Id"].unique()[:5]

for uid in sample_users:
    user_data = daily_worn[daily_worn["Id"] == uid].sort_values("date")
    ax.plot(user_data["date"], user_data["TotalSteps"], marker="o", alpha=0.7, label=str(uid))

ax.set_title("Daily Steps Over Time (5 Sample Users)")
ax.set_ylabel("Total Steps")
ax.set_xlabel("Date")
ax.legend(title="User ID", fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 8: Spread of active minutes (Q8 — Box plot)

**What this does:** shows the range, median and outliers of total active minutes per day across all users, in one compact chart.

**How it works:** a box plot draws a box from the 25th to 75th percentile, a line at the median, "whiskers" out to the normal range, and individual dots for outliers.

**Why this matters:** it's a quick way to say "half of all days had between X and Y active minutes" without listing every number.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
sns.boxplot(data=daily_worn, y="total_active_minutes", ax=ax, color="#8FBBD9")
ax.set_title("Spread of Total Active Minutes per Day")
ax.set_ylabel("Total Active Minutes")
plt.show()

print(daily_worn["total_active_minutes"].describe().round(1))

## Step 9: Sleep duration distribution (Q9 — Histogram)

**What this does:** shows how many hours of sleep people typically log per night.

**Reminder:** `sleep` only covers 24 of the 33 users — mention this whenever you present a sleep finding.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(sleep["hours_asleep"], bins=15, kde=True, ax=ax, color="#9467BD")
ax.axvline(sleep["hours_asleep"].median(), color="red", linestyle="--", label="Median")
ax.set_title("Distribution of Nightly Sleep Duration")
ax.set_xlabel("Hours Asleep")
ax.legend()
plt.show()

print(sleep["sleep_category"].value_counts())

## Step 10: Time asleep vs. time in bed (Q10 — Scatter plot)

**What this does:** visualizes sleep efficiency directly — points close to the diagonal line mean the person fell asleep almost immediately and stayed asleep; points further below the line mean more time in bed was spent awake.

**How it works:** a plain scatter, plus a reference diagonal (`ax.plot([0,max],[0,max])`) representing "100% efficiency" (asleep = in bed) for visual comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(data=sleep, x="hours_in_bed", y="hours_asleep", alpha=0.6, ax=ax)

max_val = max(sleep["hours_in_bed"].max(), sleep["hours_asleep"].max())
ax.plot([0, max_val], [0, max_val], color="red", linestyle="--", label="100% efficiency")

ax.set_title("Time Asleep vs Time in Bed")
ax.legend()
plt.show()

## Step 11: Sleep on weekdays vs weekends (Q11 — Violin plot)

**What this does:** compares not just the average but the full *shape* of the sleep-duration distribution on weekdays vs weekends. A violin plot is a box plot with the distribution's shape drawn around it, so you can see if weekend sleep is more spread out, not just longer on average.

**Finding:** weekend nights average about **7.26h** vs **6.88h** on weekdays — people sleep a bit longer and slightly more variably on weekends.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.violinplot(data=sleep, x="is_weekend", y="hours_asleep", ax=ax, palette=["#4C72B0", "#DD8452"])
ax.set_xticklabels(["Weekday", "Weekend"])
ax.set_title("Sleep Duration: Weekday vs Weekend")
ax.set_xlabel("")
plt.show()

print(sleep.groupby("is_weekend")["hours_asleep"].mean().round(2))

## Step 12: Sleep vs. next-day activity (Q12 — Scatter plot, joined data)

**What this does:** checks whether a good night's sleep leads to a more active *next* day. This is the first chart that **joins two tables**.

**How it works**
- We take each sleep record's date and add **1 day** to get `next_date`.
- We then join to `daily_worn` where the activity's `date` equals that `next_date`, matching on the same `Id`.
- This only works for date pairs where both a sleep record and the next day's activity record exist, so the joined table is smaller than either original table.

**Finding:** correlation is about **-0.16** — a weak negative relationship, meaning there's no strong evidence in this data that more sleep leads to a more active next day. Worth stating plainly in the report rather than overselling it.

In [ ]:
sleep_for_join = sleep[["Id", "date", "hours_asleep"]].copy()
sleep_for_join["next_date"] = sleep_for_join["date"] + pd.Timedelta(days=1)

sleep_next_activity = sleep_for_join.merge(
    daily_worn[["Id", "date", "TotalSteps"]],
    left_on=["Id", "next_date"], right_on=["Id", "date"],
    suffixes=("_sleep", "_activity")
)

print("Matched sleep -> next-day records:", sleep_next_activity.shape[0])

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=sleep_next_activity, x="hours_asleep", y="TotalSteps", alpha=0.5, ax=ax)
ax.set_title("Sleep Hours vs Next-Day Steps")
ax.set_xlabel("Hours Asleep (previous night)")
ax.set_ylabel("Steps (next day)")
plt.show()

print("Correlation:", round(sleep_next_activity["hours_asleep"].corr(sleep_next_activity["TotalSteps"]), 3))

## Step 13: BMI distribution (Q13 — Histogram, per user)

**What this does:** shows the spread of BMI **across the 8 users who logged weight**, using each user's average BMI (not each individual record), because 2 users contribute most of the raw records (see the cleaning notebook).

**How it works:** `groupby("Id")["BMI"].mean()` collapses each user's repeated entries down to one number.

**Reminder:** with only 8 users, this histogram is illustrative, not statistically robust — say so if you present it.

In [ ]:
bmi_per_user = weight.groupby("Id")["BMI"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(bmi_per_user, bins=8, ax=ax, color="#55A868")
ax.set_title("Average BMI per User (n=8 users)")
ax.set_xlabel("BMI")
plt.show()

print(bmi_per_user.round(1))

## Step 14: BMI category breakdown (Q14 — Pie chart, per user)

**What this does:** categorizes each user's average BMI using the same WHO bands as the cleaning notebook, then shows the split.

**Finding (n=8 users):** 4 Overweight, 3 Normal, 1 Obese, 0 Underweight.

In [ ]:
def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

bmi_category_per_user = bmi_per_user.apply(bmi_category)
category_counts = bmi_category_per_user.value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(category_counts, labels=category_counts.index, autopct=lambda p: f"{int(round(p/100*category_counts.sum()))}",
       colors=["#F1A340", "#8FBBD9", "#D7301F", "#55A868"])
ax.set_title("BMI Category Breakdown (per user, n=8)")
plt.show()

print(category_counts)

## Step 15: Weight trend per user (Q15 — Line chart)

**What this does:** tracks how each user's logged weight changed over the month.

**How it works:** grouping by `Id` and looping draws one line per user; `if len(grp) > 3` skips users with only 1-2 entries, since a "trend" needs at least a few points to mean anything.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for uid, grp in weight.groupby("Id"):
    if len(grp) > 3:
        grp_sorted = grp.sort_values("date")
        ax.plot(grp_sorted["date"], grp_sorted["WeightKg"], marker="o", label=str(uid))

ax.set_title("Weight Trend Over Time (users with 4+ logged entries)")
ax.set_ylabel("Weight (kg)")
ax.set_xlabel("Date")
ax.legend(title="User ID", fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 16: Correlation heatmap (Q16 — Heatmap, joined data)

**What this does:** puts steps, calories, active minutes, sleep and BMI side by side to see which pairs move together.

**How it works**
- Join `daily_worn` with sleep on `Id` + `date` (left join, so days without a sleep record still stay, just with a blank sleep value).
- Join in each user's average BMI on `Id`.
- `.corr()` computes every pairwise correlation, and `sns.heatmap(annot=True)` colors and labels each cell.

**Finding:** steps and active minutes correlate at about **0.71** (expected — they measure similar things). BMI correlates negatively with steps (**-0.47**) and active minutes (**-0.58**) — in this small sample, higher-BMI users tended to be less active. With only 8 BMI users this is suggestive, not conclusive.

In [ ]:
merged_for_corr = daily_worn.merge(sleep[["Id", "date", "hours_asleep"]], on=["Id", "date"], how="left")
merged_for_corr = merged_for_corr.merge(bmi_per_user.reset_index(), on="Id", how="left")

corr_cols = ["TotalSteps", "Calories", "total_active_minutes", "hours_asleep", "BMI"]
corr_matrix = merged_for_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Correlation: Activity, Sleep and BMI")
plt.tight_layout()
plt.show()

## Step 17: Hourly step pattern (Q17 — Area/line chart)

**What this does:** shows what time of day people are most active, averaged across all users and all days.

**How it works:** grouping the hourly table by `hour` (0-23) and averaging steps gives one value per hour; `fill_between` shades the area under the line to make the peak more visually obvious.

**Finding:** activity rises sharply from 6 AM, peaks around **6-7 PM** (roughly 550-600 avg steps/hour), and drops off overnight — a fairly typical daytime activity pattern.

In [ ]:
avg_steps_by_hour = hourly.groupby("hour")["StepTotal"].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(avg_steps_by_hour.index, avg_steps_by_hour.values, alpha=0.3, color="#4C72B0")
ax.plot(avg_steps_by_hour.index, avg_steps_by_hour.values, color="#4C72B0", marker="o", markersize=3)
ax.set_title("Average Steps by Hour of Day (all users, all days)")
ax.set_xlabel("Hour (0-23)")
ax.set_ylabel("Average Steps")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## Step 18: Weekday vs weekend hourly pattern (Q18 — Grouped line chart)

**What this does:** overlays two hourly curves — weekday and weekend — to see if the *timing* of activity shifts (e.g. a later, lazier start on weekends).

**How it works:** grouping by both `is_weekend` and `hour`, then `.unstack(0)` turns the weekend flag into two separate columns so both lines plot on the same axes automatically.

In [ ]:
hourly_by_weekend = hourly.groupby(["is_weekend", "hour"])["StepTotal"].mean().unstack(0)
hourly_by_weekend.columns = ["Weekday", "Weekend"]

fig, ax = plt.subplots(figsize=(10, 4))
hourly_by_weekend.plot(ax=ax, marker="o", markersize=3)
ax.set_title("Hourly Step Pattern: Weekday vs Weekend")
ax.set_xlabel("Hour (0-23)")
ax.set_ylabel("Average Steps")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## Step 19: Engagement gap across data types (Q19 — Bar chart)

**What this does:** makes the data-limitation point visual instead of just stating it in text — a chart showing how few users engage with sleep and weight logging compared to activity tracking.

**Finding:** 33 users track daily activity, only 24 log sleep, and only 8 log weight — a clear engagement drop-off that's directly relevant to Bellabeat's product/marketing strategy.

In [ ]:
total_users = daily["Id"].nunique()
engagement = pd.Series({
    "Daily Activity": total_users,
    "Sleep": sleep["Id"].nunique(),
    "Weight / BMI": weight["Id"].nunique(),
})

fig, ax = plt.subplots(figsize=(6, 4))
bars = engagement.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("User Engagement by Data Type")
ax.set_ylabel("Number of Users")
for i, v in enumerate(engagement):
    ax.text(i, v + 0.5, str(v), ha="center")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 20: Most vs. least active users (Q20 — Horizontal bar chart)

**What this does:** ranks all 33 users by their average daily steps, from least to most active — useful for identifying who might respond to different marketing messages (e.g. highly active users vs. sedentary ones).

**How it works:** `.sort_values()` orders the users, and a horizontal bar chart (`barh`) is used instead of vertical because there are 33 labels — much easier to read sideways.

In [ ]:
avg_steps_per_user = daily_worn.groupby("Id")["TotalSteps"].mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 10))
avg_steps_per_user.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Users Ranked by Average Daily Steps")
ax.set_xlabel("Average Steps")
plt.tight_layout()
plt.show()

## Summary of EDA findings

- **Activity:** average steps and calories show a moderate positive correlation (~0.57). Tuesday/Saturday are the most active days, Sunday the least.
- **Intensity:** roughly 80% of tracked time is sedentary — active minutes are a small slice of the day for most users.
- **Sleep:** median sleep is close to the recommended range, but 100 of 410 logged nights were under 6 hours. Weekend sleep is slightly longer than weekday sleep. No strong evidence that sleep predicts next-day activity (corr ≈ -0.16).
- **BMI:** based on only 8 users, most fall in Normal/Overweight. Higher BMI is associated with lower activity in this small sample (corr ≈ -0.47 to -0.58), but the sample is too small to generalize.
- **Time of day:** activity peaks in the early evening (6-7 PM) on both weekdays and weekends.
- **Engagement:** a clear drop-off from activity tracking (33 users) to sleep (24) to weight logging (8) — a finding relevant to Bellabeat's product strategy.

**Next:** `sql/analysis_queries.sql`, which turns several of these same questions into SQL queries against `db/fitbit.db`, and the Streamlit app, which will make these charts interactive.